In [140]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# from fg_data_profiling import ProfileReport

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


from sklearn.linear_model import LinearRegression, Ridge 
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import(
    RandomForestRegressor,
    GradientBoostingRegressor, 
    VotingRegressor, 
    StackingRegressor
    )

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


import warnings
warnings.filterwarnings("ignore")

In [141]:
# import sweetviz as sv 

In [142]:

df = pd.read_csv("/home/noir/Documents/GITHUB_REPOS/MACHINE_LEARNING/Data/bangladesh_student_performance.csv")
df.columns = df.columns.str.lower()

# change gender age address famsize relationship into st_(column_name)
df.columns = df.columns.str.replace('gender', 'st_gender')
df.columns = df.columns.str.replace('age', 'st_age')
df.columns = df.columns.str.replace('address', 'st_address')
df.columns = df.columns.str.replace('famsize', 'st_famsize')
df.columns = df.columns.str.replace('relationship', 'st_relationship')

df.head()

,date,st_gender,st_age,st_address,st_famsize,pstatus,m_edu,f_edu,m_job,f_job,st_relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
0,29/04/2018,M,18,Rural,GT3,Together,3,2,At_home,Farmer,No,No,71672,4,4.22,3.72
1,29/04/2018,F,19,Rural,LE3,Apart,0,4,Other,Health,Yes,No,26085,5,3.47,2.62
2,29/04/2018,F,19,Rural,GT3,Together,0,3,Teacher,Services,No,No,40891,3,3.32,2.56
3,29/04/2018,F,19,Rural,LE3,Apart,2,3,At_home,Business,No,No,50600,2,4.57,4.17
4,29/04/2018,M,17,Rural,GT3,Together,1,1,At_home,Farmer,No,No,62458,2,4.50,3.94


In [143]:
# report = sv.analyze(df)
# report.show_html("sweet_report.html")

In [144]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             2018 non-null   object 
 1   st_gender        2018 non-null   object 
 2   st_age           2018 non-null   int64  
 3   st_address       2018 non-null   object 
 4   st_famsize       2018 non-null   object 
 5   pstatus          2018 non-null   object 
 6   m_edu            2018 non-null   int64  
 7   f_edu            2018 non-null   int64  
 8   m_job            2018 non-null   object 
 9   f_job            2018 non-null   object 
 10  st_relationship  2018 non-null   object 
 11  smoker           2018 non-null   object 
 12  tuition_fee      2018 non-null   int64  
 13  time_friends     2018 non-null   int64  
 14  ssc_result       2018 non-null   float64
 15  hsc_result       2018 non-null   float64
dtypes: float64(2), int64(5), object(9)
memory usage: 252.4+ KB


In [145]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
st_age,2018.0,17.981169,0.826340,17.0,17.00,18.00,19.00,19.0
m_edu,2018.0,1.871160,1.194206,0.0,1.00,2.00,3.00,4.0
f_edu,2018.0,2.174430,1.252979,0.0,1.00,2.00,3.00,4.0
tuition_fee,2018.0,72977.637760,24045.222595,25102.0,53619.50,71272.50,90904.75,129168.0
time_friends,2018.0,3.059960,1.439190,1.0,2.00,3.00,4.00,5.0
ssc_result,2018.0,3.788087,0.622376,2.0,3.36,3.77,4.23,5.0
hsc_result,2018.0,3.199177,0.604526,2.0,2.78,3.16,3.58,5.0


In [146]:
df.isnull().sum()

date               0
st_gender          0
st_age             0
st_address         0
st_famsize         0
pstatus            0
m_edu              0
f_edu              0
m_job              0
f_job              0
st_relationship    0
smoker             0
tuition_fee        0
time_friends       0
ssc_result         0
hsc_result         0
dtype: int64

In [147]:
df.duplicated().sum()

np.int64(0)

In [148]:
df["date"].nunique()
df.drop("date", axis=1, inplace=True)

In [149]:
df.shape

(2018, 15)

In [150]:
# detecting colums 

categorical_cols = df.select_dtypes(include="object").columns
numerical_cols = df.select_dtypes(include=np.number).columns

# lower case all column names

print("Categorical Columns:", categorical_cols.tolist())
print("Numerical Columns:", numerical_cols.tolist())

Categorical Columns: ['st_gender', 'st_address', 'st_famsize', 'pstatus', 'm_job', 'f_job', 'st_relationship', 'smoker']
Numerical Columns: ['st_age', 'm_edu', 'f_edu', 'tuition_fee', 'time_friends', 'ssc_result', 'hsc_result']


In [151]:
# Correlation Analysis
corr_target = df[numerical_cols].corr()['hsc_result'].sort_values(ascending=False)
corr_target 

hsc_result      1.000000
ssc_result      0.950178
m_edu           0.063776
f_edu           0.054811
tuition_fee     0.038068
st_age         -0.009857
time_friends   -0.156356
Name: hsc_result, dtype: float64

In [152]:
# from data_profiling import ProfileReport

# profile = ProfileReport(df, title="Bangladesh Student Performance Profiling Report", explorative=True)
# profile.to_file("bangladesh_student_performance_profile.html")

In [153]:
X = df.drop("hsc_result", axis=1)
y = df["hsc_result"]

# PipeLine


In [163]:
# For numerical Features 

numerical_transformer = Pipeline(
    steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
    ]
)

# For categorical features
categorical_transformer = Pipeline(
    steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)


# Preprocessor 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols.drop("hsc_result")),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [156]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [165]:
# Base-learner 
reg = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42)
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
xgb = XGBRegressor(n_estimators=100, random_state=42)


In [166]:
voting_regressor = VotingRegressor(
    estimators=[
        ("lr", reg),
        ("rf", rf),
        ("gb", gb),
        ("xgb", xgb)
    ]
)

In [167]:
stacking_regressor = StackingRegressor(
    estimators=[
        ("lr", reg),
        ("rf", rf),
        ("gb", gb),
        ("xgb", xgb)
    ],
    
    final_estimator= Ridge() # the meta learner
)

In [168]:
# Dictionary of models

models = {
    "LinearRegression": reg,
    "RandomForestRegressor": rf,
    "GradientBoostingRegressor": gb,
    "XGBRegressor": xgb,
    "VotingRegressor": voting_regressor,
    "StackingRegressor": stacking_regressor
}



In [169]:
# Training and Evaluation

results = []

for name, model in models.items():
    pipeline = Pipeline(
        [
            ('preprocessor', preprocessor),
            ('model', model)
        ]
    )
    
    # train 
    pipeline.fit(X_train, y_train)
    
    # predict 
    y_pred = pipeline.predict(X_test) 
    
    # evaluate
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    results.append({
        "Model": name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2_Score": r2
    })
    
results_df = pd.DataFrame(results)
results_df.sort_values(by="R2_Score", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)


print(results_df)

                       Model       MSE      RMSE       MAE  R2_Score
0  GradientBoostingRegressor  0.015155  0.123107  0.098902  0.959565
1          StackingRegressor  0.015302  0.123703  0.099143  0.959172
2            VotingRegressor  0.016248  0.127469  0.102413  0.956649
3      RandomForestRegressor  0.018647  0.136556  0.108201  0.950248
4           LinearRegression  0.020269  0.142371  0.111376  0.945920
5               XGBRegressor  0.020773  0.144128  0.116065  0.944577


In [170]:
best_model_name = results_df.loc[0, "Model"]
print(f"Best Model: {best_model_name}")

Best Model: GradientBoostingRegressor
